# Conversational E-commerce Assistant — end-to-end demo

Runs the real code of this repository in **deterministic fallback mode** (no API key, no Docker, no ML extras): keyword intent router, template synthesizer, hash embeddings, lexical reranker, in-memory stores. With `ANTHROPIC_API_KEY` / `VOYAGE_API_KEY` / `COHERE_API_KEY` in the environment the same cells use the production components.

Run from the repository root with the `.venv` kernel: `pip install -e ".[dev]"`.

In [1]:
import os
for k, v in {'EMBEDDING_BACKEND': 'hash', 'RERANK_BACKEND': 'lexical', 'VECTOR_STORE': 'in_memory', 'LOG_LEVEL': 'WARNING'}.items():
    os.environ.setdefault(k, v)
from src.config import get_settings
get_settings.cache_clear()
s = get_settings()
print('model:', s.anthropic_model, '| llm_enabled:', s.llm_enabled, '| embeddings:', s.embedding_backend, '| rerank:', s.rerank_backend, '| store:', s.vector_store)

model: claude-sonnet-4-5-20250929 | llm_enabled: False | embeddings: hash | rerank: lexical | store: in_memory


## 1. Build the runtime (catalog, BM25, dense store, cart, session cache) and compile the graph

In [2]:
from src.agents.runtime import build_runtime
from src.agents.orchestrator import build_graph
from src.observability import configure_logging
configure_logging('WARNING')
rt = await build_runtime()
rt.graph = build_graph(rt)
print(len(rt.catalog), 'products |', rt.dense.name, '|', rt.embedder.name, '|', rt.reranker.name)

213 products | in_memory | hash | lexical


## 2. Hybrid retrieval: BM25 + dense + RRF, then rerank + filters

In [3]:
from src.retrieval.hybrid import HybridRetriever
from src.retrieval.structured import parse_filters, apply_filter, lexically_related
retriever = HybridRetriever(rt.bm25, rt.dense, rt.embedder)
for q in ['Heinz ketchup 397 gram bottle', 'top rated cereals under $5', 'cheapest coffee']:
    hits = await retriever.search(q, k=20)                    # BM25 + dense + RRF
    pool = await rt.reranker.rerank(q, hits, top_n=10)         # rerank
    f = parse_filters(q)                                       # price / rating / sort
    top = pool[:5] if f.is_empty() else (apply_filter(lexically_related(q, pool), f) or apply_filter(lexically_related(q, hits), f))[:5]
    print(q, '| filters:', f.model_dump(exclude_defaults=True) or 'none')
    for p in top:
        print(f'   [pid:{p.product_id}] {p.product_name} · {p.aisle} · ${p.price_usd} · rating {p.avg_rating}')

Heinz ketchup 397 gram bottle | filters: none
   [pid:1] Heinz Tomato Ketchup 397g · pasta sauce · $4.29 · rating 4.6
   [pid:2] Heinz Tomato Ketchup 567g Squeeze · pasta sauce · $5.99 · rating 4.7
   [pid:114] Minestrone · soup · $13.22 · rating 4.32
   [pid:124] Whole Bean Colombian 12oz · coffee · $10.77 · rating 4.48
   [pid:123] Chamomile Tea 20ct · tea · $9.42 · rating 3.81
top rated cereals under $5 | filters: {'price_max': 5.0, 'sort': 'rating_desc'}
   [pid:4] Cheerios Honey Nut Kid-Friendly 12oz · cereal · $4.79 · rating 4.7
cheapest coffee | filters: {'sort': 'price_asc'}
   [pid:124] Whole Bean Colombian 12oz · coffee · $10.77 · rating 4.48
   [pid:126] Cold Brew Concentrate · coffee · $21.44 · rating 4.35
   [pid:125] Espresso Ground 10oz · coffee · $24.37 · rating 4.12


## 3. A six-turn conversation through the LangGraph agent

The session state (history + last shown products) is threaded between turns exactly as the API does it.

In [4]:
from src.agents.orchestrator import run_turn, next_session_state
from src.session.cache import SessionState
session = SessionState()
for msg in ['hi', 'lactose free milk gallon', 'add 2 of the second one to my cart', 'add the almond milk too', 'remove the almond milk from my cart', 'refund the milk']:
    resp, state_out = await run_turn(rt.graph, session_id='nb', user_id='demo-user', message=msg, session=session)
    session = next_session_state(session, resp, state_out)
    cart = [(c.product_id, c.quantity) for c in resp.cart_snapshot]
    print(f'[{resp.turn}] {msg!r}\n    intent={resp.intent} escalated={resp.escalated} cart={cart} latency={resp.latency_ms}ms cost=${resp.cost_usd}\n    -> {resp.response}')

[1] 'hi'
    intent=greeting escalated=False cart=[] latency=4ms cost=$0.0
    -> Hi! What can I help you find today?
[2] 'lactose free milk gallon'
    intent=product_search escalated=False cart=[] latency=16ms cost=$0.0
    -> Top matches for your query: [pid:6] Lactose-Free 2% Milk 1 Gallon ($5.49), [pid:32] Lactose-Free Milk ($22.80), [pid:8] Lactose-Free Whole Milk 64oz ($4.29).
[3] 'add 2 of the second one to my cart'
    intent=cart_op escalated=False cart=[(32, 2)] latency=2ms cost=$0.0
    -> Added 2 x Lactose-Free Milk to your cart. Cart: [pid:32] Lactose-Free Milk x 2. Total $45.60.
[4] 'add the almond milk too'
    intent=cart_op escalated=False cart=[(32, 2), (30, 1)] latency=3ms cost=$0.0
    -> Added 1 x Almond Milk Unsweetened to your cart. Cart: [pid:32] Lactose-Free Milk x 2; [pid:30] Almond Milk Unsweetened x 1. Total $62.45.
[5] 'remove the almond milk from my cart'
    intent=cart_op escalated=False cart=[(32, 2)] latency=3ms cost=$0.0
    -> Removed Almond Milk Un

## 4. Refund policy and escalation

In [5]:
for msg in ['refund my $200 order from last week', 'refund order #1003', 'I want to speak to a human agent']:
    resp, _ = await run_turn(rt.graph, session_id='nb-policy', user_id='demo-user', message=msg)
    print(f'{msg!r} -> intent={resp.intent} escalated={resp.escalated}\n    {resp.response}')

'refund my $200 order from last week' -> intent=refund escalated=True
    I'm escalating this to a human teammate (Refund of $200.00 needs a human: value $200.00 exceeds auto-refund threshold $100.00.) — they'll follow up shortly.
'refund order #1003' -> intent=refund escalated=True
    I'm escalating this to a human teammate (Refund of $41.30 needs a human: sensitive category: alcohol.) — they'll follow up shortly.
'I want to speak to a human agent' -> intent=escalate escalated=True
    I'm escalating this to a human teammate (customer asked for a human) — they'll follow up shortly.


## 5. Retrieval ablation on the hand-labelled eval set (subset for speed)

`python -m eval.run` runs the full 100 queries and regenerates `eval/RESULTS.md`; the report below is labelled with the components that produced it.

In [6]:
from src.eval.runner import evaluate, load_eval_set, format_report
subset = [q for q in load_eval_set() if q['category'] in ('lookup', 'comparative')][:30]
report = await evaluate('in_memory', eval_set=subset, embedder=rt.embedder, reranker=rt.reranker)
print(format_report(report))

E-COMMERCE RETRIEVAL EVAL  (store: in_memory, n=30)
components: fallback determinista (embedding por hash / rerank lexico), sin modelo

Strategy       P@5     R@10     MRR    nDCG@10    p95_ms
------------------------------------------------------------------------------
bm25         0.300    1.000   1.000      0.995       0.3
dense        0.300    0.983   1.000      0.988      18.0
hybrid       0.300    0.983   1.000      0.989      17.4
rerank       0.293    0.983   1.000      0.992       0.3

By category:
  [lookup]
    bm25     P@5=0.30  R@10=1.00  MRR=1.00  nDCG@10=0.99
    dense    P@5=0.30  R@10=0.98  MRR=1.00  nDCG@10=0.99
    hybrid   P@5=0.30  R@10=0.98  MRR=1.00  nDCG@10=0.99
    rerank   P@5=0.29  R@10=0.98  MRR=1.00  nDCG@10=0.99



## 6. Multi-turn task completion (deterministic checks)

In [7]:
from src.eval.conversation import evaluate_conversations
records = [r for r in load_eval_set() if r['category'] == 'multi_turn'][:8]
out = await evaluate_conversations(records, rt)
print('label:', out['label'])
print('task completion:', out['task_completion_rate'], '| avg turns:', out['avg_turns'], '| p95 ms:', round(out['latency_p95_ms'], 1))
for c in out['conversations']:
    print(' ', c['id'], 'PASS' if c['passed'] else 'FAIL ' + str(c['failed_checks']))
await rt.close()

label: fallback determinista, sin LLM (router heuristico + plantillas)
task completion: 1.0 | avg turns: 2.625 | p95 ms: 38.6
  q-multiturn-01 PASS
  q-multiturn-02 PASS
  q-multiturn-03 PASS
  q-multiturn-04 PASS
  q-multiturn-05 PASS
  q-multiturn-06 PASS
  q-multiturn-07 PASS
  q-multiturn-08 PASS
